# 🪄 AUTO SOLVER — ปุ่มเดียวจบ (All-in-one สำหรับคนที่อยากง่ายที่สุด)

โน้ตบุ๊กนี้ "เดาเอง" ว่าโจทย์เป็นงานอะไร แล้วเลือกวิธี + เทรน + สร้าง submission ให้อัตโนมัติ
คุณแก้แค่ `ที่เดียว` (ชื่อ competition หรืออัปโหลดไฟล์) แล้วกดรันทีละเซลล์จากบนลงล่าง

ในนี้มีให้ครบ: ดูโจทย์ (metric/ฟอร์แมต) -> ดูข้อมูล (EDA) -> ส่ง baseline ก่อน -> เทรนจริง + ดูคะแนนในเครื่อง -> รวมไฟล์ (blend) -> ส่ง
ทำอัตโนมัติได้: จำแนกรูป / จำแนกข้อความ / ตาราง-จำแนก/ทำนายตัวเลข (รวม multiclass แบบส่ง prob)
ทำให้ไม่ได้ (จะบอกให้ไปเปิดโน้ตบุ๊กเฉพาะ): ตัดคำ, segmentation, บรรยายรูป/VQA, detection, เสียง(audio)


## ขั้นที่ 1 — ติดตั้ง (กดรันรอ ~5-10 นาทีครั้งแรก)

In [ ]:
!pip -q install -U "autogluon.tabular[all]" "autogluon.multimodal" pandas numpy pillow

## ขั้นที่ 2 — เอาข้อมูลเข้า (เลือก 1 ใน 3 วิธี) + เช็ค GPU

เซลล์ล่างรองรับ 3 วิธี แก้แค่ตัวแปรบนสุดให้ตรงกับวิธีที่ใช้:

วิธี A (แนะนำ) ดาวน์โหลดจาก Kaggle อัตโนมัติ: ต้องมี `kaggle.json` (kaggle.com -> รูปโปรไฟล์ -> Settings -> Account -> Create New Token)
- บน `Kaggle Notebook`: ข้อมูลต่อไว้ที่ `/kaggle/input/...` แล้ว ไม่ต้องใส่ token
- บน `Colab/เครื่อง`: ใส่ `KAGGLE_USERNAME` + `KAGGLE_KEY`

วิธี B โหลดข้อมูลมาเครื่องตัวเองแล้วอัปขึ้น Colab เอง: ลากไฟล์ (เช่น `data.zip`) ไปวางในแถบ Files ซ้ายมือของ Colab
แล้วตั้ง `DATA_DIR = "/content"` (หรือโฟลเดอร์ที่วาง) -> เซลล์จะแตก zip ให้เอง ไม่ต้องใช้ token

วิธี C ต่อ Google Drive: รัน `from google.colab import drive; drive.mount('/content/drive')` ก่อน แล้วตั้ง `DATA_DIR = "/content/drive/MyDrive/โฟลเดอร์ของคุณ"`

เซลล์นี้เช็ค GPU ให้ด้วย ถ้างานเป็น deep learning (รูป/ข้อความ/สัญญาณดิบ) ควรเปิด GPU: เมนู `Runtime > Change runtime type > T4 GPU`

In [ ]:
import os, glob, subprocess

# ----- วิธี A: ดาวน์โหลดจาก Kaggle -----
COMP = "ใส่-slug-ของ-competition-ตรงนี้"      # << แก้: slug ท้าย URL เช่น kaggle.com/competitions/titanic -> "titanic"  (ใช้เฉพาะวิธี A)
KAGGLE_USERNAME = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "peetwan1"  (บน Kaggle Notebook เว้นว่างได้)
KAGGLE_KEY      = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "0a1b2c3d4e5f..." (คีย์ยาว ~32 ตัวจาก kaggle.json)
# ----- วิธี B/C: อัปโหลดเอง / ต่อ Google Drive -----
DATA_DIR = ""          # << แก้: ถ้าอัปโหลดข้อมูลเอง ใส่ path โฟลเดอร์ เช่น "/content" (วิธี B) หรือ "/content/drive/MyDrive/data" (วิธี C) ; ใช้ Kaggle (วิธี A) ปล่อยว่าง

# เช็ค GPU (ถ้าไม่มี + งานเป็น deep learning -> เปิดที่เมนู Runtime > Change runtime type > T4 GPU)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี (งาน deep learning จะช้า; งานตาราง/ตัดคำ ไม่ต้องใช้ GPU ก็ได้)")
except Exception:
    pass

def get_data(comp):
    # วิธี B/C: ผู้ใช้ตั้ง DATA_DIR เอง (อัปโหลด/ต่อ Drive) -- แตก zip ในโฟลเดอร์นั้นให้ด้วยถ้ามี
    if DATA_DIR and os.path.isdir(DATA_DIR):
        for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
            subprocess.run(["unzip", "-o", "-q", z, "-d", DATA_DIR])
        print("ใช้ข้อมูลจากโฟลเดอร์ที่ตั้งเอง:", DATA_DIR); return DATA_DIR
    # บน Kaggle Notebook ข้อมูลต่อไว้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle ข้อมูลอยู่ที่", kpath); return kpath
    # วิธี A: ดาวน์โหลดด้วย Kaggle API
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("\nไฟล์ที่มี (ดูชื่อไฟล์/โฟลเดอร์ แล้วแก้ในเซลล์ถัดไปให้ตรง):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — ดูโจทย์ (ต้องส่งอะไร/metric) + ดูข้อมูล (EDA) + เดาประเภทงาน

รันแล้วมันบอกเอง: ต้องเติมคอลัมน์อะไร, metric อะไร (ดึงจาก Kaggle), ข้อมูลหน้าตาเป็นไง, และเดาว่าเป็นงานอะไร
ถ้าเดาผิด ค่อยตั้ง `FORCE_TASK` / `METRIC` ที่บรรทัดบนสุด

In [ ]:
import os, glob, random, pandas as pd, numpy as np
IMG_EXT = (".jpg",".jpeg",".png",".bmp")
AUDIO_EXT = (".wav",".mp3",".flac",".ogg",".m4a")

# ===== ปรับเฉพาะถ้า auto เดาผิด (ปกติปล่อยไว้) =====
FORCE_TASK = None    # << ใส่ได้: "image" / "text" / "tabular" / "regression"
METRIC     = None    # << ใส่ "roc_auc" ถ้าโจทย์วัด AUC (จะได้ส่งความน่าจะเป็น)

def seed_everything(s=42):   # ตั้ง seed ให้ผลรันซ้ำได้
    random.seed(s); np.random.seed(s); os.environ["PYTHONHASHSEED"]=str(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except Exception: pass
seed_everything(42)

def _find(name):
    h = glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True); return h[0] if h else None
def _dir_with(exts, keyword=None):
    cand = [d for d,_,fs in os.walk(DATA_DIR) if any(f.lower().endswith(exts) for f in fs)
            and (keyword is None or keyword in d.lower())]
    return max(cand, key=lambda d: len(os.listdir(d))) if cand else None

sample = pd.read_csv(_find("sample_submission.csv"))
ID_COL, ANS_COL = sample.columns[0], sample.columns[1]
train = pd.read_csv(_find("train.csv")) if _find("train.csv") else None
test  = pd.read_csv(_find("test.csv"))  if _find("test.csv")  else None
train_img, test_img = _dir_with(IMG_EXT, "train"), _dir_with(IMG_EXT, "test")
has_audio = _dir_with(AUDIO_EXT) is not None

# 🔎 ต้องส่งอะไร + metric อะไร
print("ไฟล์ส่งต้องมีคอลัมน์:", list(sample.columns), "| จำนวนแถว:", len(sample))
for _c in list(sample.columns)[1:]:
    print(f"  - เติม '{_c}': ชนิด {sample[_c].dtype}, ตัวอย่าง = {list(sample[_c].dropna().unique()[:3])}")
_metric = None
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    _api = KaggleApi(); _api.authenticate()
    _resp = _api.competitions_list(search=COMP)
    for _co in (getattr(_resp, "competitions", _resp) or []):
        if str(getattr(_co, "ref", "")).rstrip("/").split("/")[-1] == COMP:
            _metric = getattr(_co, "evaluation_metric", None) or getattr(_co, "evaluationMetric", None); break
except Exception:
    pass
_ml = (_metric or "").lower()
WANT_PROB = (METRIC in ("roc_auc","auc")) or any(k in _ml for k in ["auc","roc","log loss","logloss","brier","probab"])
MULTICOL  = len(sample.columns) > 2
print("Metric (จาก Kaggle):", _metric or "ดึงไม่ได้ (เปิด tab Evaluation อ่านเอง)",
      "| auto จะส่งเป็น:", "ความน่าจะเป็น" if (WANT_PROB or MULTICOL) else "ป้าย/ตัวเลข")

# 👀 EDA สั้น ๆ
if train is not None:
    print("\nEDA: train", train.shape, "| แถวซ้ำ", int(train.duplicated().sum()))
    _miss = (train.isnull().mean()*100).round(1); _miss = _miss[_miss > 0]
    if len(_miss): print("  คอลัมน์มีค่าว่าง(%):", _miss.to_dict())
    if test is not None:
        print("  คอลัมน์มีใน train ไม่มีใน test:", (set(train.columns)-set(test.columns)) or "(ไม่มี)")

# 🚧 ดักงานที่ AUTO ทำไม่ได้ -> บอกให้ไปโน้ตบุ๊กเฉพาะ
if has_audio and not (train_img and test_img):
    raise SystemExit("งานนี้เป็นเสียง (audio) -> เปิด 06_Audio/audio_classification.ipynb")
assert train is not None, "หา train.csv ไม่เจอ -> เช็ค path / ใช้โน้ตบุ๊กเฉพาะ"
if test is not None and len(sample) != len(test) and not (train_img and test_img):
    raise SystemExit("submission ไม่ใช่ 1 แถวต่อ 1 แถวของ test (ตัดคำ/segmentation) -> เปิด 00_MASTER_ROUTER เลือกโน้ตบุ๊กเฉพาะ")
assert (test is not None) or (train_img and test_img), \
    "หา test.csv ไม่เจอ (อาจชื่ออื่น) -> เปลี่ยนชื่อเป็น test.csv หรือใช้โน้ตบุ๊กเฉพาะ"

def detect_target():
    if test is not None:
        cand = [c for c in train.columns if c not in test.columns and c != ID_COL]
        if len(cand) == 1: return cand[0]
    if ANS_COL in train.columns: return ANS_COL
    return [c for c in train.columns if c != ID_COL][-1]
def detect_task():
    if FORCE_TASK: return FORCE_TASK
    if train_img and test_img:
        ic = [c for c in train.columns if train[c].astype(str).str.lower().str.endswith(IMG_EXT).mean() > 0.5]
        if ic: return "image"
    tc = [c for c in train.columns if train[c].dtype == object and c != ID_COL and train[c].astype(str).str.len().mean() >= 10]
    if tc: return "text"
    tgt = detect_target()
    if pd.api.types.is_numeric_dtype(train[tgt]) and train[tgt].nunique() > 20: return "regression"
    return "tabular"
task = detect_task()
print("\nตรวจพบว่าเป็นงาน:", {"image":"จำแนกรูป","text":"จำแนกข้อความ","tabular":"ตาราง-จำแนกคลาส",
                          "regression":"ตาราง-ทำนายตัวเลข"}.get(task, task))

## ขั้นที่ 4 — ส่ง baseline ก่อน (สำคัญมาก!)

เซลล์นี้สร้าง submission แบบโง่ ๆ (ทายคลาสที่เจอบ่อยสุด / ค่าเฉลี่ย) ใน ~0 วินาที
ส่งอันนี้ "ก่อน" เทรนจริง เพื่อ (1) ล็อกว่าฟอร์แมตถูก ส่งได้ (2) ได้คะแนนพื้นไว้ก่อน กันพลาดเสียสิทธิ์ส่ง
วิธีส่ง: ไปขั้นสุดท้าย เปลี่ยน `FILE = "submission_baseline.csv"` แล้วรัน

In [ ]:
base = sample.copy()
if (not MULTICOL) and (ANS_COL in train.columns):
    _t = train[ANS_COL]
    base[ANS_COL] = _t.mean() if (pd.api.types.is_numeric_dtype(_t) and _t.nunique() > 20) else _t.mode()[0]
base.to_csv("submission_baseline.csv", index=False)
assert list(base.columns) == list(sample.columns), "ฟอร์แมต baseline ไม่ตรง sample"
print("บันทึก submission_baseline.csv -> แนะนำส่งอันนี้ก่อนเลย")
print("ค่าที่เติม:", base[ANS_COL].iloc[0] if ANS_COL in base else "(หลายคอลัมน์ = 0)")

## ขั้นที่ 5 — AUTO เทรนจริง + สร้าง submission + ดูคะแนนในเครื่อง

มันจะ print "คะแนน validation ในเครื่อง" ให้ด้วย จะได้รู้ว่าดีพอจะส่งไหม โดยไม่ต้องเปลือง quota

In [ ]:
out = sample.copy()
if task == "image":
    from autogluon.multimodal import MultiModalPredictor
    IMG_COL = [c for c in train.columns if train[c].astype(str).str.lower().str.endswith(IMG_EXT).mean() > 0.5][0]
    LABEL_COL = [c for c in train.columns if c not in (IMG_COL, ID_COL)][0]
    exts = [os.path.splitext(f)[1] for f in os.listdir(test_img) if f.lower().endswith(IMG_EXT)]
    ext = exts[0] if exts else ".jpg"
    print("  รูป:", IMG_COL, "| ป้าย:", LABEL_COL, "| นามสกุล test:", ext)
    tr = train.copy(); tr["image"] = tr[IMG_COL].apply(lambda n: os.path.join(train_img, str(n)))
    p = MultiModalPredictor(label=LABEL_COL, path="ag_auto").fit(tr[["image", LABEL_COL]], time_limit=900)
    td = sample.copy(); td["image"] = td[ID_COL].apply(lambda i: os.path.join(test_img, str(i)+ext))
    out[ANS_COL] = p.predict(td[["image"]]).values
elif task == "text":
    from autogluon.multimodal import MultiModalPredictor
    TEXT_COL = [c for c in train.columns if train[c].dtype == object and c != ID_COL and train[c].astype(str).str.len().mean() >= 10][0]
    LABEL_COL = detect_target(); print("  ข้อความ:", TEXT_COL, "| ป้าย:", LABEL_COL)
    p = MultiModalPredictor(label=LABEL_COL, path="ag_auto").fit(train[[TEXT_COL, LABEL_COL]], time_limit=600)
    out[ANS_COL] = p.predict(test[[TEXT_COL]]).values
else:
    from autogluon.tabular import TabularPredictor
    LABEL_COL = detect_target(); is_reg = (task == "regression")
    print("  เป้าหมาย:", LABEL_COL, "| ชนิด:", "ทำนายตัวเลข" if is_reg else "จำแนกคลาส")
    tr = train.drop(columns=[c for c in [ID_COL] if c in train.columns])
    te = test.drop(columns=[c for c in [ID_COL] if c in test.columns])
    p = TabularPredictor(label=LABEL_COL, problem_type=("regression" if is_reg else None),
                         path="ag_auto").fit(tr, presets="best_quality", time_limit=600)
    if is_reg:
        out[ANS_COL] = p.predict(te).values
    elif MULTICOL:
        proba = p.predict_proba(te); name_map = {str(cc): cc for cc in proba.columns}
        for col in list(sample.columns[1:]):
            key = name_map.get(str(col)) or name_map.get(str(col).split("_")[-1])
            if key is None: print("  เตือน: จับคู่คอลัมน์", col, "กับคลาสไม่ได้ -> เติม 0 (ตรวจชื่อคอลัมน์)")
            out[col] = proba[key].values if key is not None else 0.0
        print("  (เติมความน่าจะเป็นต่อคลาส", len(sample.columns)-1, "คอลัมน์)")
    elif WANT_PROB and getattr(p, "positive_class", None) is not None:
        out[ANS_COL] = p.predict_proba(te)[p.positive_class].values
    else:
        out[ANS_COL] = p.predict(te).values
        if getattr(p, "positive_class", None) is not None and train[LABEL_COL].nunique() == 2:
            o2 = sample.copy(); o2[ANS_COL] = p.predict_proba(te)[p.positive_class].values
            o2.to_csv("submission_proba.csv", index=False); print("  (เซฟ submission_proba.csv เผื่อ metric เป็น AUC)")

# คะแนน validation ในเครื่อง (ไม่ต้องเดา/ไม่เปลือง quota ส่ง)
try:
    _lb = p.leaderboard()
    print("  คะแนน validation (ในเครื่อง) ดีสุด ~", round(float(_lb["score_val"].max()), 4))
except Exception:
    pass
out.to_csv("submission.csv", index=False)
print("\nเสร็จ! บันทึก submission.csv", out.shape)
try: display(out.head())
except Exception: print(out.head())

## ขั้นที่ 6 — รวมหลายไฟล์ (blend) ดันคะแนนฟรี (ไม่บังคับ)

ถ้าทำหลายวิธี/หลายโมเดล (มีหลายไฟล์ submission) การเฉลี่ยรวมกันมักได้คะแนนเพิ่มแบบไม่ต้องเทรนใหม่
เซลล์นี้รวมไฟล์ `submission*.csv` ด้วย rank-average (ดีสุดสำหรับความน่าจะเป็น)

In [ ]:
import glob
FILES = [f for f in glob.glob("submission*.csv") if "baseline" not in f and "blend" not in f]   # << เลือกไฟล์ที่จะรวม
print("ไฟล์ที่จะรวม:", FILES)
if len(FILES) >= 2:
    dfs = [pd.read_csv(f) for f in FILES]; bl = dfs[0].copy()
    for col in bl.columns[1:]:
        if pd.api.types.is_numeric_dtype(bl[col]):
            bl[col] = sum(d[col].rank(pct=True) for d in dfs) / len(dfs)   # rank-avg
    bl.to_csv("submission_blend.csv", index=False)
    print("บันทึก submission_blend.csv -> ลองส่งเทียบกับ submission.csv")
else:
    print("มีไฟล์เดียว ยังไม่ต้อง blend (ทำหลายวิธีก่อนแล้วค่อยกลับมารวม)")

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
import pandas as pd, glob, os
FILE = "submission.csv"   # << แก้เป็นไฟล์ที่จะส่ง เช่น "submission_lgbm.csv" หรือ "submission_timm.csv"
# ตรวจรูปแบบก่อนส่งอัตโนมัติ (กันแก้ config ผิดแล้วส่งฟอร์แมตเพี้ยน -> ได้ 0 คะแนน)
_sub = pd.read_csv(FILE)
_sam = glob.glob(os.path.join(DATA_DIR, "*ample*ubmission*.csv"))
if _sam:
    _s = pd.read_csv(_sam[0])
    print("คอลัมน์ตรง sample:", list(_sub.columns)==list(_s.columns), "| จำนวนแถวตรง:", len(_sub)==len(_s))
    assert list(_sub.columns)==list(_s.columns), f"คอลัมน์ไม่ตรง sample! {list(_sub.columns)} != {list(_s.columns)} -> แก้ก่อนส่ง"
print("พร้อมส่ง:", FILE, _sub.shape)
!kaggle competitions submit -c {COMP} -f {FILE} -m "auto solver"
!kaggle competitions submissions -c {COMP} | head